In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from skimage.metrics import structural_similarity as ssim, mean_squared_error
from skimage.metrics import peak_signal_noise_ratio as psnr
from sewar.full_ref import uqi, msssim


def calculate_fsim(original, processed):
    original_gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    processed_gray = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
    feature_similarity = ssim(original_gray, processed_gray, full=True)[0]
    return feature_similarity

def load_dataset(dataset_path):
    labels = os.listdir(dataset_path)
    images, targets = [], []
    for label in labels:
        folder_path = os.path.join(dataset_path, label)
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is not None:
                images.append(cv2.resize(img, (224, 224)))  # Resize for uniformity
                targets.append(label)
    return np.array(images), np.array(targets)

# Gaussian Amended Bilateral Filter
def gaussian_amended_bilateral_filter(image, d=5, sigma_color=75, sigma_space=75):
    return cv2.bilateralFilter(image, d, sigma_color, sigma_space)

# Savitzky-Golay Filter
def savitzky_golay_filter(image):
    return cv2.GaussianBlur(image, (5, 5), 0)

# AmPel Optimization
def ampel_optimization(image, d_range, sigma_color_range, sigma_space_range, iterations=10, population_size=20):
    population = [
        {"d": random.randint(*d_range),
         "sigma_color": random.uniform(*sigma_color_range),
         "sigma_space": random.uniform(*sigma_space_range)}
        for _ in range(population_size)
    ]
    best_params = None
    best_score = float('-inf')

    for _ in range(iterations):
        for candidate in population:
            # Terapkan GABF dengan parameter kandidat
            filtered_image = gaussian_amended_bilateral_filter(
                image, d=candidate["d"],
                sigma_color=candidate["sigma_color"],
                sigma_space=candidate["sigma_space"]
            )
            # Evaluasi menggunakan PSNR
            score = psnr(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY),
                         cv2.cvtColor(filtered_image, cv2.COLOR_BGR2GRAY))

            if score > best_score:
                best_score = score
                best_params = candidate
        for candidate in population:
            candidate["d"] = int((candidate["d"] + best_params["d"]) / 2)
            candidate["sigma_color"] = (candidate["sigma_color"] + best_params["sigma_color"]) / 2
            candidate["sigma_space"] = (candidate["sigma_space"] + best_params["sigma_space"]) / 2

    return best_params, best_score

def opt_cfa(image):
    param_ranges = {"d_range": (3, 9), "sigma_color_range": (50, 100), "sigma_space_range": (50, 100)}
    best_params, _ = ampel_optimization(image, **param_ranges)

    # Terapkan GABF dengan parameter terbaik
    stage1 = gaussian_amended_bilateral_filter(
        image, d=best_params["d"],
        sigma_color=best_params["sigma_color"],
        sigma_space=best_params["sigma_space"]
    )
    stage2 = savitzky_golay_filter(stage1)
    return stage2

def visualize_preprocessing(original, processed, title):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title(f'Citra Asli - {title}')
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.subplot(1, 2, 2)
    plt.title(f'Citra Setelah Diproses - {title}')
    plt.imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    plt.show()

def evaluate_metrics(original, processed):
    original_gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    processed_gray = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
    mse = mean_squared_error(original_gray, processed_gray)
    psnr_value = psnr(original_gray, processed_gray)
    ssim_value = ssim(original_gray, processed_gray)
    uqi_value = uqi(original_gray, processed_gray)
    fsim_value = calculate_fsim(original, processed)
    msssim_value = msssim(original_gray, processed_gray)
    return mse, psnr_value, ssim_value, uqi_value, fsim_value, msssim_value

if _name_ == "_main_":
    dataset_path = "Cotton leaf Dataset"
    images, labels = load_dataset(dataset_path)
    sub_folders = os.listdir(dataset_path)
    sample_images = []
    for sub_folder in sub_folders:
        folder_path = os.path.join(dataset_path, sub_folder)
        img_name = random.choice(os.listdir(folder_path))  # Randomly pick one image
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is not None:
            sample_images.append((sub_folder, img))

    for sub_folder, img in sample_images:
        processed_img = opt_cfa(img)
        metrics = evaluate_metrics(img, processed_img)
        print(f"Sub-folder: {sub_folder}")
        print(f"MSE: {metrics[0]:.2f}, PSNR: {metrics[1]:.2f}, SSIM: {metrics[2]:.2f}")
        print(f"UQI: {metrics[3]:.2f}, FSIM: {metrics[4]:.2f}, MS-SSIM: {metrics[5]:.2f}")
        visualize_preprocessing(img, processed_img, sub_folder)